# Coiled Benchmark — Remote Dask Workers Near the Data

**The idea:** Cal-Adapt's Zarr stores live on S3 in `us-west-2`. Our laptop is far away.
Coiled spins up Dask workers *inside* `us-west-2` — they pull data from S3 at datacenter
speed (~10 Gbps), process it, and send back only the tiny averaged results.

**Same test parameters as other benchmarks:**
- 1 region (Joshua Tree), 1 scenario (Historical), 5 years (2010-2014)
- 3 variables: T_Max, T_Min, Precip
- Full pipeline through cos-weighted spatial average
- Results saved to CSV for comparison with `test_output.csv`

**Prerequisites:**
- `pip install coiled` (done)
- `coiled login` (done)
- Software environment `caladapt-env` created (done)

## 1. Local Setup 

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import time
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Paths ---
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PARK_OUTLINES_DIR = os.path.join(PROJECT_ROOT, "parkOutlines")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Load Joshua Tree boundary locally ---
jt_path = os.path.join(PARK_OUTLINES_DIR, "JoshuaTree", "Joshua_Tree_National_Park.shp")
boundary = gpd.read_file(jt_path).to_crs("EPSG:4326")
bounds = boundary.total_bounds  # [minx, miny, maxx, maxy]
lat_bounds = (bounds[1], bounds[3])
lon_bounds = (bounds[0], bounds[2])

# Serialize the boundary geometry for sending to workers
# (GeoDataFrames can't be pickled cleanly, but WKT strings can)
boundary_wkt = boundary.geometry.to_wkt().tolist()
boundary_crs = str(boundary.crs)

print(f"JT bounds: lon({bounds[0]:.2f}, {bounds[2]:.2f}) lat({bounds[1]:.2f}, {bounds[3]:.2f})")
print(f"Boundary polygons: {len(boundary_wkt)}")
print(f"Project root: {PROJECT_ROOT}")

JT bounds: lon(-116.46, -115.26) lat(33.67, 34.13)
Boundary polygons: 1
Project root: /Users/andrewschoenen/Desktop/DSE


In [2]:
# --- Load the Cal-Adapt catalog CSV (runs locally, it's tiny) ---
t0 = time.perf_counter()
catalog_url = "https://cadcat.s3.amazonaws.com/cae-zarr.csv"
catalog = pd.read_csv(catalog_url)
print(f"Catalog loaded: {len(catalog)} rows ({time.perf_counter()-t0:.1f}s)")

# Filter to LOCA2 monthly 3km historical
loca2_mon = catalog[
    (catalog.activity_id == "LOCA2") &
    (catalog.table_id == "mon") &
    (catalog.grid_label == "d03")
].copy()

VARIABLE_MAP = {
    "T_Max":  "tasmax",
    "T_Min":  "tasmin",
    "Precip": "pr",
}
EXPERIMENT = "historical"

# Build S3 path lists per variable
paths_by_var = {}
for vk, vid in VARIABLE_MAP.items():
    rows = loca2_mon[
        (loca2_mon.variable_id == vid) &
        (loca2_mon.experiment_id == EXPERIMENT)
    ]
    paths_by_var[vk] = rows[["source_id", "member_id", "path"]].to_dict("records")
    print(f"  {vk:6s} ({vid:6s}): {len(paths_by_var[vk])} Zarr stores")

Catalog loaded: 8572 rows (0.3s)
  T_Max  (tasmax): 70 Zarr stores
  T_Min  (tasmin): 70 Zarr stores
  Precip (pr    ): 70 Zarr stores


## 2. Start Coiled Cluster

In [3]:
import coiled
from dask.distributed import Client

print("Starting Coiled cluster in us-west-2 (same region as Cal-Adapt S3)...")
print("This may take 1-2 minutes on first run (provisioning VMs).")
print()

t0 = time.perf_counter()

cluster = coiled.Cluster(
    name="caladapt-test",
    software="caladapt-env",
    region="us-west-2",           # MUST match cadcat S3 bucket
    n_workers=4,                   # 4 workers for this small test
    worker_memory="8 GiB",         # 2 vCPU + 8 GiB = standard balanced instance
    spot_policy="spot_with_fallback",  # Cheaper
    idle_timeout="15 minutes",     # Auto-shutdown
)

client = cluster.get_client()
t_startup = time.perf_counter() - t0

print(f"\nCluster ready in {t_startup:.0f}s!")
print(f"  Workers:   {len(client.scheduler_info()['workers'])}")
print(f"  Dashboard: {client.dashboard_link}")
print(f"\nOpen the dashboard link above to watch workers in real-time.")

Starting Coiled cluster in us-west-2 (same region as Cal-Adapt S3)...
This may take 1-2 minutes on first run (provisioning VMs).



[2026-02-04 11:47:32,848][INFO    ][coiled] Creating Cluster (name: caladapt-test, https://cloud.coiled.io/clusters/1419674 ). This usually takes 1-2 minutes...



Cluster ready in 158s!
  Workers:   3
  Dashboard: https://cluster-qljfe.dask.host/FNCGYq4qV-ZZSxaI/status

Open the dashboard link above to watch workers in real-time.


/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/distributed/client.py:1596: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.3.5  | 2.4.2     | 2.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


## 3. Define Processing Functions

These functions will be sent to and executed on the remote workers.
They use only packages in `caladapt-env` (xarray, rioxarray, numpy, etc.).

In [4]:
import dask


@dask.delayed
def process_variable_remote(var_key, var_id, paths_list, time_slice_start, time_slice_end,
                            lat_bounds, lon_bounds, boundary_wkt, boundary_crs):
    """
    Runs ENTIRELY on a Coiled worker in us-west-2.
    Opens Zarr stores from S3, loads, clips, averages.
    Returns a pandas DataFrame (small, sent back to laptop).
    """
    import xarray as xr
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import rioxarray
    from shapely import wkt
    import fsspec
    import time as _time
    
    t0 = _time.perf_counter()
    time_sel = slice(str(time_slice_start), str(time_slice_end))
    
    # --- Reconstruct boundary from WKT ---
    geometries = [wkt.loads(w) for w in boundary_wkt]
    boundary_gdf = gpd.GeoDataFrame(geometry=geometries, crs=boundary_crs)
    
    # --- Open all Zarr stores for this variable ---
    datasets = []
    for info in paths_list:
        store = fsspec.get_mapper(info["path"], anon=True)
        ds = xr.open_zarr(store, consolidated=True)
        da = ds[var_id]
        
        # Slice time and space
        da = da.sel(time=time_sel)
        lat_dim = "lat" if "lat" in da.dims else "latitude"
        lon_dim = "lon" if "lon" in da.dims else "longitude"
        da = da.sel(
            **{lat_dim: slice(lat_bounds[0], lat_bounds[1]),
               lon_dim: slice(lon_bounds[0], lon_bounds[1])}
        )
        
        sim_name = f"LOCA2_{info['source_id']}_{info['member_id']}"
        da = da.expand_dims(simulation=[sim_name])
        datasets.append(da)
    
    t_open = _time.perf_counter() - t0
    
    # --- Concat all simulations ---
    combined = xr.concat(datasets, dim="simulation")
    
    # --- Unit conversion ---
    if var_key in ("T_Max", "T_Min"):
        combined = combined - 273.15  # K → °C
    elif var_key == "Precip":
        days = combined.time.dt.days_in_month
        combined = combined * 86400 * days  # kg/m²/s → mm/month
    
    # --- CRS setup ---
    if combined.rio.crs is None:
        combined = combined.rio.write_crs("EPSG:4326")
    lat_dim = "lat" if "lat" in combined.dims else "latitude"
    lon_dim = "lon" if "lon" in combined.dims else "longitude"
    combined = combined.rio.set_spatial_dims(x_dim=lon_dim, y_dim=lat_dim)
    
    # --- Load into memory (FAST because worker is in us-west-2!) ---
    t_load_start = _time.perf_counter()
    loaded = combined.load()
    t_load = _time.perf_counter() - t_load_start
    
    # --- Clip to park boundary ---
    masked = loaded.rio.clip(boundary_gdf.geometry.values, all_touched=True, drop=False)
    
    # --- Cos-weighted spatial average ---
    weights = np.cos(np.deg2rad(masked[lat_dim]))
    weights.name = "weights"
    avg = masked.weighted(weights).mean(dim=[lon_dim, lat_dim], skipna=True)
    
    t_total = _time.perf_counter() - t0
    
    # --- Convert to DataFrame for transfer back to laptop ---
    df = avg.to_dataframe(name=var_key).reset_index()
    df["_open_time"] = t_open
    df["_load_time"] = t_load
    df["_total_time"] = t_total
    
    return df

print("Remote processing function defined.")

Remote processing function defined.


## 4. Benchmark

| # | Strategy | Where it runs | Description |
|---|----------|---------------|-------------|
| I | Coiled sequential | 1 worker at a time | Baseline for remote |
| J | Coiled parallel (3 workers) | 3 workers simultaneously | All 3 vars in parallel |

Compare against Strategy E (climakitae local) from TestDirectS3: ~480s

In [5]:
# ================================================================
# STRATEGY I: Coiled sequential — one variable at a time
# ================================================================

print("=" * 65)
print("STRATEGY I: Coiled sequential (1 var at a time)")
print("=" * 65)

results_i = {}
t0 = time.perf_counter()
for vk, vid in VARIABLE_MAP.items():
    t_start = time.perf_counter()
    
    # Create delayed task
    delayed_df = process_variable_remote(
        var_key=vk,
        var_id=vid,
        paths_list=paths_by_var[vk],
        time_slice_start=2010,
        time_slice_end=2014,
        lat_bounds=lat_bounds,
        lon_bounds=lon_bounds,
        boundary_wkt=boundary_wkt,
        boundary_crs=boundary_crs,
    )
    
    # Submit to cluster and wait for result
    df = delayed_df.compute()
    
    # Extract timing from worker
    open_t = df["_open_time"].iloc[0]
    load_t = df["_load_time"].iloc[0]
    total_t = df["_total_time"].iloc[0]
    
    results_i[vk] = df.drop(columns=["_open_time", "_load_time", "_total_time"])
    wall = time.perf_counter() - t_start
    print(f"  {vk:6s}: {wall:5.1f}s wall  (worker: open={open_t:.1f}s load={load_t:.1f}s total={total_t:.1f}s)")

t_i = time.perf_counter() - t0
print(f"  TOTAL: {t_i:.1f}s\n")

STRATEGY I: Coiled sequential (1 var at a time)
  T_Max :  27.8s wall  (worker: open=17.9s load=7.5s total=25.6s)
  T_Min :  23.2s wall  (worker: open=18.4s load=4.2s total=22.8s)
  Precip:  25.0s wall  (worker: open=16.5s load=7.8s total=24.5s)
  TOTAL: 76.0s



In [6]:
# ================================================================
# STRATEGY J: Coiled parallel — all 3 vars at once on separate workers
# ================================================================

print("=" * 65)
print("STRATEGY J: Coiled parallel (3 vars on 3 workers)")
print("=" * 65)

t0 = time.perf_counter()

# Create all delayed tasks
delayed_tasks = {}
for vk, vid in VARIABLE_MAP.items():
    delayed_tasks[vk] = process_variable_remote(
        var_key=vk,
        var_id=vid,
        paths_list=paths_by_var[vk],
        time_slice_start=2010,
        time_slice_end=2014,
        lat_bounds=lat_bounds,
        lon_bounds=lon_bounds,
        boundary_wkt=boundary_wkt,
        boundary_crs=boundary_crs,
    )

# Submit ALL at once — Dask distributes across workers
computed = dask.compute(*delayed_tasks.values())

results_j = {}
for vk, df in zip(delayed_tasks.keys(), computed):
    open_t = df["_open_time"].iloc[0]
    load_t = df["_load_time"].iloc[0]
    total_t = df["_total_time"].iloc[0]
    results_j[vk] = df.drop(columns=["_open_time", "_load_time", "_total_time"])
    print(f"  {vk:6s}: worker open={open_t:.1f}s  load={load_t:.1f}s  total={total_t:.1f}s")

t_j = time.perf_counter() - t0
print(f"  TOTAL (wall clock): {t_j:.1f}s\n")

STRATEGY J: Coiled parallel (3 vars on 3 workers)
  T_Max : worker open=15.4s  load=11.4s  total=27.0s
  T_Min : worker open=15.5s  load=11.3s  total=26.9s
  Precip: worker open=14.0s  load=14.1s  total=28.1s
  TOTAL (wall clock): 28.4s



In [7]:
# ================================================================
# ACCURACY CHECK — compare against climakitae test_output.csv
# ================================================================

print("=" * 65)
print("ACCURACY CHECK")
print("=" * 65)

# I vs J should be identical
all_ij_match = True
for vk in VARIABLE_MAP:
    # Merge on simulation + time
    merged = results_i[vk].merge(
        results_j[vk], on=["simulation", "time"], suffixes=("_I", "_J")
    )
    if len(merged) > 0:
        col_i = f"{vk}_I"
        col_j = f"{vk}_J"
        match = np.allclose(merged[col_i].values, merged[col_j].values, equal_nan=True)
        print(f"  {vk:6s}  I==J: {match}  (matched {len(merged)} rows)")
        if not match:
            all_ij_match = False
    else:
        print(f"  {vk:6s}  No matching rows between I and J!")
        all_ij_match = False

print(f"\n  I/J identical: {all_ij_match}")

# Compare T_Max against test_output.csv (from climakitae)
old_csv = os.path.join(OUTPUT_DIR, "test_output.csv")
if os.path.exists(old_csv):
    df_old = pd.read_csv(old_csv)
    df_coiled = results_j["T_Max"].copy()
    df_coiled["time"] = pd.to_datetime(df_coiled["time"]).dt.strftime("%Y-%m-%d")
    
    merged = df_old.merge(df_coiled, on=["simulation", "time"], suffixes=("_ck", "_coiled"))
    if len(merged) > 0:
        diff = np.abs(merged["TMax"] - merged["T_Max"])
        close = np.allclose(merged["TMax"].values, merged["T_Max"].values, rtol=1e-4)
        print(f"\n  T_Max vs climakitae test_output.csv:")
        print(f"    Matched rows:      {len(merged)}")
        print(f"    Max abs diff:      {diff.max():.6f} °C")
        print(f"    Mean abs diff:     {diff.mean():.6f} °C")
        print(f"    np.allclose:       {close}")
    else:
        print(f"\n  No matching rows — check simulation name format:")
        print(f"    Old: {df_old['simulation'].iloc[:2].tolist()}")
        print(f"    New: {df_coiled['simulation'].iloc[:2].tolist()}")
else:
    print(f"\n  No test_output.csv found — run TestNotebook first.")

ACCURACY CHECK
  T_Max   I==J: True  (matched 4200 rows)
  T_Min   I==J: True  (matched 4200 rows)
  Precip  I==J: True  (matched 4200 rows)

  I/J identical: True

  T_Max vs climakitae test_output.csv:
    Matched rows:      4200
    Max abs diff:      0.000013 °C
    Mean abs diff:     0.000001 °C
    np.allclose:       True


In [8]:
# ================================================================
# SUMMARY TABLE
# ================================================================

# Reference timings from other benchmarks (fill in your actuals)
LOCAL_BASELINE = 480.5  # Strategy E from TestDirectS3 (climakitae sequential)

print()
print("=" * 65)
print(f"{'Strategy':<50s} {'Time':>6s}  {'vs local':>8s}")
print("-" * 65)
print(f"{'E) Local climakitae sequential (reference)':<50s} {LOCAL_BASELINE:5.1f}s  {'1.0x':>8s}")
print(f"{'I) Coiled sequential (1 worker)':<50s} {t_i:5.1f}s  {LOCAL_BASELINE/t_i:6.1f}x")
print(f"{'J) Coiled parallel   (3 workers)':<50s} {t_j:5.1f}s  {LOCAL_BASELINE/t_j:6.1f}x")
print("-" * 65)
best = min(t_i, t_j)
best_label = {t_i: "I", t_j: "J"}[best]
print(f"FASTEST: Strategy {best_label} ({best:.1f}s) — {LOCAL_BASELINE/best:.1f}x vs local baseline")
print(f"Cluster startup: ~{t_startup:.0f}s (one-time cost per session)")
print(f"I/J results identical: {all_ij_match}")
print("=" * 65)


Strategy                                             Time  vs local
-----------------------------------------------------------------
E) Local climakitae sequential (reference)         480.5s      1.0x
I) Coiled sequential (1 worker)                     76.0s     6.3x
J) Coiled parallel   (3 workers)                    28.4s    16.9x
-----------------------------------------------------------------
FASTEST: Strategy J (28.4s) — 16.9x vs local baseline
Cluster startup: ~158s (one-time cost per session)
I/J results identical: True


## 5. CSV Export — Sanity Check

In [9]:
# Save T_Max from Coiled Strategy J in same format as test_output.csv
df_export = results_j["T_Max"].copy()
df_export = df_export.rename(columns={"T_Max": "TMax"})
df_export["time"] = pd.to_datetime(df_export["time"]).dt.strftime("%Y-%m-%d")
df_export["Year"] = pd.to_datetime(df_export["time"]).dt.year
df_export["Month"] = pd.to_datetime(df_export["time"]).dt.month
df_export["Region"] = "JoshuaTree"
df_export["Variable"] = "T_Max"
df_export["Scenario"] = "Historical Climate"

export_cols = ["time", "simulation", "TMax", "Year", "Month", "Region", "Variable", "Scenario"]
df_export = df_export[export_cols].sort_values(["simulation", "time"]).reset_index(drop=True)

output_file = os.path.join(OUTPUT_DIR, "test_coiled_output.csv")
df_export.to_csv(output_file, index=False)
print(f"Saved: {output_file}")
print(f"  Rows: {len(df_export)}")
print(f"  Simulations: {df_export['simulation'].nunique()}")
print(f"\nFirst 5 rows:")
df_export.head()

Saved: /Users/andrewschoenen/Desktop/DSE/data/csv/test_coiled_output.csv
  Rows: 4200
  Simulations: 70

First 5 rows:


,time,simulation,TMax,Year,Month,Region,Variable,Scenario
0,2010-01-01,LOCA2_ACCESS-CM2_r1i1p1f1,16.409840,2010,1,JoshuaTree,T_Max,Historical Climate
1,2010-02-01,LOCA2_ACCESS-CM2_r1i1p1f1,15.929316,2010,2,JoshuaTree,T_Max,Historical Climate
2,2010-03-01,LOCA2_ACCESS-CM2_r1i1p1f1,21.021435,2010,3,JoshuaTree,T_Max,Historical Climate
3,2010-04-01,LOCA2_ACCESS-CM2_r1i1p1f1,22.410185,2010,4,JoshuaTree,T_Max,Historical Climate
4,2010-05-01,LOCA2_ACCESS-CM2_r1i1p1f1,28.960201,2010,5,JoshuaTree,T_Max,Historical Climate


## 6. Shutdown Cluster

**Important:** Always shut down the cluster when done to avoid charges.
The `idle_timeout` will also auto-shutdown after 15 min of inactivity.

In [10]:
cluster.close()
print("Cluster shut down.")
print("You can also check active clusters at: https://cloud.coiled.io")

[2026-02-04 11:51:54,003][INFO    ][coiled] Cluster 1419674 deleted successfully.


Cluster shut down.
You can also check active clusters at: https://cloud.coiled.io


## Summary

**How this compares to our other benchmarks:**

| Strategy | Time | Speedup | Notes |
|----------|------|---------|-------|
| A: Local climakitae sequential | ~480s | 1x | Baseline |
| H: Local direct S3 + cache (repeat) | ~9s | 53x | But 3.8 GB cache for 5 years |
| I: Coiled sequential | ?s | ?x | Remote, no cache needed |
| J: Coiled parallel | ?s | ?x | Remote, 3 workers |

**For the full pipeline** (150 years × 4 scenarios × 3 vars × 2 parks):
- Scale up to 10-20 workers
- Process all 24 (region, scenario) combinations in parallel
- Each worker pulls data at datacenter speed
- Only final CSVs come back to your laptop
- No local disk cache needed